# 第9章　Transformer 入门 — 注意力机制（Attention）与 LLM 的本质

ChatGPT 等大语言模型（LLM）的内核就是 **Transformer**，其心脏是**注意力机制（self-attention）**。
本章**亲手实现 attention 建立直觉**，用 PyTorch 现成部件搭 Transformer，最后引到真正的 LLM（Hugging Face）的入口。

目标：理解 attention 公式 `softmax(QKᵀ/√d)V` 的含义，能跑通一个小型 Transformer 分类器。

> **使用方法**：从上到下 `Shift + Enter`。推荐 GPU 的章节（Colab：代码执行程序→更改运行时类型→GPU）。

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 9-1. 为什么用 Transformer？

- 文本是**词的序列**。"它"指代谁，可能依赖很远的词（长距离依赖）。
- 旧的 RNN 逐词顺序处理 → 慢、远距离依赖差。
- **Transformer 一次性看整个序列，直接计算词与词的关系**。可并行、擅长长距离。
- 把这个机制用海量数据＋超大模型训练出来的，就是 LLM。

## 9-2. 注意力（Attention）的直觉：Q・K・V

计算"每个词该**关注（attend）**其他哪些词、从而获取信息"。用检索来比喻：

- **Query（查询 Q）**＝想查什么（当前词"想知道什么"）
- **Key（键 K）**＝每个词的标题（"我有这类信息"）
- **Value（值 V）**＝每个词的内容（实际传递的信息）

算 Q 与各个 K 的**相关度（内积）** → 用 softmax 变成权重（和为1）→ 用该权重对 V 做**加权平均**。
这样就能"关系越近的词，信息混入越多"。公式：

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V$$

除以 $\sqrt{d}$ 是为了防止内积过大让 softmax 过于尖锐（缩放）。

## 9-3. 亲手实现 attention
用小张量确认权重"和为1"，并得到 V 的加权平均。

In [ ]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    d = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d ** 0.5)   # (..., L, L) 每对词的相关度
    weights = F.softmax(scores, dim=-1)             # 每行和为1
    out = weights @ V                               # 用权重对 V 加权平均
    return out, weights

torch.manual_seed(0)
L, d = 4, 8                       # 序列长4, 维度8
Q = torch.randn(L, d); K = torch.randn(L, d); V = torch.randn(L, d)
out, w = scaled_dot_product_attention(Q, K, V)
print("attention权重 (每行和=1):")
print(w.round(decimals=2))
print("每行之和:", w.sum(dim=-1))   # 全是 1
print("输出 shape:", out.shape)     # (4, 8)

## 9-4. PyTorch 现成部件：`nn.MultiheadAttention`

实际会把 Q・K・V 分成多个"头"并行算注意力（**多头注意力**）。PyTorch 已内置。
`batch_first=True` 让形状变成 `(批, 序列长, 维度)`，更直观。

In [ ]:
import torch.nn as nn
mha = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
x = torch.randn(2, 5, 16)            # (批2, 序列长5, 维度16)
attn_out, attn_w = mha(x, x, x)      # self-attention: Q=K=V=x
print("输出:", attn_out.shape)        # (2, 5, 16)
print("注意力权重:", attn_w.shape)     # (2, 5, 5)

## 9-5. Transformer 块与位置信息

一个块 = **多头注意力 → 全连接（FFN）**，各自带**残差连接＋归一化**。PyTorch 里就是一个 `nn.TransformerEncoderLayer`。

重要：**attention 不知道顺序**（把词打乱结果也一样）。
所以要给每个位置**加上位置信息（positional encoding/embedding）**告诉它"这是第几个"。

In [ ]:
layer = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=64, batch_first=True)
encoder = nn.TransformerEncoder(layer, num_layers=2)   # 叠2个块
x = torch.randn(2, 5, 16)
print("Transformer输出:", encoder(x).shape)            # (2, 5, 16)

## 9-6. 小型 Transformer 分类器（端到端）

玩具任务：长度8的数字序列（0〜9），**含有"7"就判为 1，否则 0**。试试 attention 能否"在序列里找到7"。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
N, L, VOCAB = 4000, 8, 10
X = torch.randint(0, VOCAB, (N, L))
y = (X == 7).any(dim=1).long()                 # 含7则为1
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
print("标签为1的比例:", y.float().mean().item())

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab=10, d=32, seq_len=8, nhead=4, nlayers=2, nclass=2):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)       # 数字 -> 向量
        self.pos = nn.Embedding(seq_len, d)     # 位置 -> 向量（可学习的位置嵌入）
        layer = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=64, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, nlayers)
        self.head = nn.Linear(d, nclass)
    def forward(self, x):                       # x: (N, L) 整数序列
        L = x.size(1)
        pos = torch.arange(L, device=x.device).unsqueeze(0)   # (1, L)
        h = self.tok(x) + self.pos(pos)         # 词嵌入 + 位置嵌入
        h = self.enc(h)                         # (N, L, d)
        h = h.mean(dim=1)                       # 沿序列平均（池化）
        return self.head(h)                     # (N, 2)

model = TinyTransformer()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

for epoch in range(8):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X).argmax(1) == y).float().mean().item()
    print(f"epoch {epoch+1}: loss={loss.item():.4f}  acc={acc:.3f}")

若几个 epoch 后准确率接近 1.0，就说明 Transformer 通过 attention 学到了"序列中某处有没有7"这种**与位置无关的关系**。

## 9-7. 走向真正的 LLM — Hugging Face（只读即可）

从零造巨型 LLM 很难，实务里都用**预训练模型**。用 `transformers` 库只要几行。
下面是示例（运行需要 `pip install transformers` 和联网）：

```python
# pip install transformers
from transformers import pipeline

# 情感分析（英文）
clf = pipeline("sentiment-analysis")
print(clf("I love learning PyTorch!"))
# -> [{'label': 'POSITIVE', 'score': 0.999...}]

# 文本生成
gen = pipeline("text-generation", model="gpt2")
print(gen("Once upon a time", max_length=30)[0]["generated_text"])
```

用自己的数据做**微调（fine-tuning）**，也就是把前面学的"训练循环＋optimizer"原样用上。

## 9-8. 下一步
- 论文「Attention Is All You Need」(2017) — Transformer 原典
- Andrej Karpathy「Let's build GPT」/ nanoGPT — 从零实现字符级 GPT（顶级教材）
- Hugging Face Course（免费）: https://huggingface.co/learn
- PyTorch 官方：`nn.Transformer` 教程

## 练习 9
1. 把玩具任务改成"**序列之和为偶数则为1**"再训练（需要全局聚合而非位置）。
2. 改 `nlayers`、`nhead`、`d`，看精度・速度变化。
3. 用 9-3 的 `scaled_dot_product_attention`，自造让注意力集中到某个词的输入，观察权重矩阵。
4. （进阶）实际跑一下 Hugging Face 的 `pipeline`，也试试中文的情感分析模型。

In [ ]:
# 在这里写你自己的代码并运行
